# MNIST EDA — Exploratory Data Analysis

Quick look at the `keras.datasets.mnist` dataset before building the baseline CNN:

1. Load the data and inspect shapes
2. Class distribution (is it balanced?)
3. Sample digit grid
4. Pixel intensity histogram


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape:  {x_test.shape}")
print(f"y_test shape:  {y_test.shape}")
print(f"pixel range:   [{x_train.min()}, {x_train.max()}]")
print(f"classes:       {sorted(np.unique(y_train).tolist())}")


## 1. Class Distribution

MNIST is close to balanced across the 10 digit classes, but not perfectly — worth
checking before assuming accuracy alone is a fair metric.


In [ ]:
labels, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(8, 5))
plt.bar(labels, counts, color="steelblue")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.title("Training Set Class Distribution")
plt.xticks(labels)
for label, count in zip(labels, counts):
    plt.text(label, count + 50, str(count), ha="center", fontsize=8)
plt.tight_layout()
plt.show()

print(f"min class count: {counts.min()}, max class count: {counts.max()}")
print(f"imbalance ratio: {counts.max() / counts.min():.3f}")


## 2. Sample Digit Grid

A quick visual sanity check — random samples with their labels.


In [ ]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(x_train), size=15, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(10, 6))
for idx, ax in zip(sample_idx, axes.flat):
    ax.imshow(x_train[idx], cmap="gray")
    ax.set_title(f"Label: {y_train[idx]}", fontsize=10)
    ax.axis("off")
plt.suptitle("Random MNIST Training Samples", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Pixel Intensity Histogram

Most pixels are pure background (0) or pure stroke (255), with a smaller mid-range
band from anti-aliased edges. This is why simple `/255.0` normalization works well —
the distribution is already bimodal and well-behaved.


In [ ]:
sample_pixels = x_train[:2000].astype("float32").flatten()

plt.figure(figsize=(8, 5))
plt.hist(sample_pixels, bins=50, color="coral", edgecolor="black", linewidth=0.3)
plt.yscale("log")
plt.xlabel("Pixel Intensity (0-255)")
plt.ylabel("Frequency (log scale)")
plt.title("Pixel Intensity Distribution (2,000 training images)")
plt.tight_layout()
plt.show()


## Takeaways

- Classes are roughly balanced (~9-11% each) — plain accuracy is a fair headline metric.
- Images are clean, centered, single-channel 28x28 grayscale digits.
- Pixel values are bimodal (background vs. stroke), so `/255.0` normalization to `[0, 1]`
  (as used in `src/data/loader.py`) is sufficient — no need for per-image standardization.
